# TG2 – Aprendizado Supervisionado: Classificação com SVM
## PUCRS – Escola Politécnica | Aprendizado Supervisionado 2026/1 – Prof. Duncan

| Matrícula | Nome |
|-----------|------|
| 24200227   | João Henrique da Luz |
| XXXXXXX   | Integrante 2 |

**Dataset:** Loan Prediction – Risk Flag  
**Objetivo:** Treinar e avaliar modelos SVM (Support Vector Machine) para prever o risco de inadimplência de empréstimos.


## 1. Instalação de dependências e importações

Instalamos `gdown` para baixar os arquivos do Google Drive diretamente no Colab.  
As bibliotecas principais utilizadas são: `pandas`, `numpy`, `scikit-learn` e `matplotlib`.


In [ ]:
# Instalação de dependências (necessário apenas no Colab)
!pip install gdown -q

import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (accuracy_score, f1_score,
                             precision_score, recall_score,
                             classification_report, ConfusionMatrixDisplay)
from sklearn.inspection import DecisionBoundaryDisplay

warnings.filterwarnings('ignore')
print("Imports concluídos com sucesso.")


## 2. Carregamento dos dados

Os arquivos de treino e teste são baixados do Google Drive.  
**Treino:** `loan_prediction_5.csv` | **Teste:** `loan_prediction_test.csv`

In [ ]:
# ===== IDs dos arquivos no Google Drive ===== #
TRAIN_FILE_ID = "https://drive.google.com/file/d/1QKQ9TzgU2LikT0moSN9B3NqQAknLo1sa/view"
TEST_FILE_ID  = "https://drive.google.com/file/d/1WHt0go-KwEjGGDHyceVxyaJmcjO0_kCa/view"

# Download dos arquivos
!gdown --id {TRAIN_FILE_ID} -O loan_prediction_5.csv
!gdown --id {TEST_FILE_ID}  -O loan_prediction_test.csv

# Leitura dos CSVs
df_train = pd.read_csv("loan_prediction_5.csv")
df_test  = pd.read_csv("loan_prediction_test.csv")

print(f"Treino: {df_train.shape[0]:,} linhas × {df_train.shape[1]} colunas")
print(f"Teste : {df_test.shape[0]:,} linhas × {df_test.shape[1]} colunas")


## 3. Análise Exploratória dos Dados (EDA)

Inspecionamos a estrutura, distribuição e qualidade dos dados antes de qualquer transformação.  
É fundamental entender o conjunto de **treino** antes de tocar no teste.


In [ ]:
# ===== Visão geral do conjunto de treino ===== #
print("=== Primeiras linhas ===")
display(df_train.head())

print("\n=== Tipos de dados e valores nulos ===")
print(df_train.info())

print("\n=== Estatísticas descritivas (numéricos) ===")
display(df_train.describe())


In [ ]:
# ── Valores nulos ─────────────────────────────────────────────────────────────
print("Valores nulos por coluna (treino):")
display(df_train.isnull().sum().to_frame("nulos"))

print("\nValores nulos por coluna (teste):")
display(df_test.isnull().sum().to_frame("nulos"))


In [ ]:
# ── Distribuição da variável alvo ─────────────────────────────────────────────
print("Distribuição Risk_Flag (treino):")
print(df_train["Risk_Flag"].value_counts())
print(f"  Proporção: {df_train['Risk_Flag'].mean():.2%} de alto risco")

fig, ax = plt.subplots(figsize=(5, 4))
df_train["Risk_Flag"].value_counts().plot(kind="bar", ax=ax,
    color=["#4CAF50", "#F44336"], edgecolor="black")
ax.set_title("Distribuição da variável alvo (Risk_Flag) – Treino")
ax.set_xlabel("Risk_Flag (0 = Baixo risco, 1 = Alto risco)")
ax.set_ylabel("Contagem")
ax.set_xticklabels(["0 – Baixo risco", "1 – Alto risco"], rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# ── Variáveis categóricas ─────────────────────────────────────────────────────
cat_cols = ["Married/Single", "House_Ownership", "Car_Ownership",
            "Profession", "CITY", "STATE"]

for col in cat_cols:
    print(f"{col}: {df_train[col].nunique()} valores únicos | "
          f"Exemplos: {list(df_train[col].unique()[:5])}")


In [ ]:
# ── Histogramas das variáveis numéricas ───────────────────────────────────────
num_cols = ["Income", "Age", "Experience", "CURRENT_JOB_YRS", "CURRENT_HOUSE_YRS"]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    for flag, color, label in [(0, "#4CAF50", "Baixo risco"), (1, "#F44336", "Alto risco")]:
        axes[i].hist(df_train[df_train["Risk_Flag"] == flag][col],
                     bins=30, alpha=0.6, color=color, label=label)
    axes[i].set_title(col)
    axes[i].set_xlabel("Valor")
    axes[i].set_ylabel("Frequência")
    axes[i].legend()

axes[-1].set_visible(False)
fig.suptitle("Distribuição das variáveis numéricas por classe (Treino)", fontsize=13)
plt.tight_layout()
plt.show()


### 3.1 Correlação dos atributos com a variável alvo

Para identificar os **2 atributos mais correlacionados** com `Risk_Flag` (necessário para o `DecisionBoundaryDisplay`),  
calculamos a correlação de Pearson após a codificação ordinal das variáveis categóricas com `LabelEncoder`.  
Esse encoder é usado **somente para análise exploratória**, não para o treinamento do modelo.


In [ ]:
from sklearn.preprocessing import LabelEncoder

# Codificação apenas para análise de correlação
df_enc = df_train.copy()
for col in cat_cols:
    le = LabelEncoder()
    df_enc[col] = le.fit_transform(df_enc[col].astype(str))

feature_cols = [c for c in df_enc.columns if c not in ["Id", "Risk_Flag"]]
corrs = df_enc[feature_cols].corrwith(df_enc["Risk_Flag"]).abs().sort_values(ascending=False)

print("=== Correlação absoluta com Risk_Flag ===")
print(corrs.to_string())

# Os dois mais correlacionados
TOP2 = corrs.index[:2].tolist()
print(f"\n✅ 2 atributos mais correlacionados com Risk_Flag: {TOP2}")

# Gráfico
fig, ax = plt.subplots(figsize=(10, 5))
corrs.plot(kind="bar", ax=ax, color="#1976D2", edgecolor="black")
ax.set_title("Correlação absoluta de cada atributo com Risk_Flag (Treino)")
ax.set_ylabel("|Correlação de Pearson|")
ax.set_xlabel("Atributo")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


### 3.2 Análise Exploratória do Conjunto de Teste

Inspecionamos o teste separadamente para verificar se há categorias não vistas no treino  
(o que exigiria tratamento especial no pipeline de pré-processamento).


In [ ]:
# ── Verificação de categorias novas no teste vs treino ────────────────────────
print("=== Verificação de valores no TESTE não presentes no TREINO ===")
for col in cat_cols:
    train_vals = set(df_train[col].astype(str).unique())
    test_vals  = set(df_test[col].astype(str).unique())
    novos = test_vals - train_vals
    if novos:
        print(f"  ⚠️  {col}: {len(novos)} valor(es) novo(s) → {novos}")
    else:
        print(f"  ✅  {col}: nenhum valor novo no teste")

print("\n=== Distribuição Risk_Flag (Teste) ===")
print(df_test["Risk_Flag"].value_counts())


## 4. Pré-processamento

Passos realizados **exclusivamente com base no treino** (sem "contaminar" com informação do teste):

1. Removemos a coluna `Id` (identificador sem valor preditivo).
2. Aplicamos **OneHotEncoder** nas variáveis categóricas.  
   - `handle_unknown='ignore'` garante que categorias novas no teste não causem erros.
3. Aplicamos **StandardScaler** nas variáveis numéricas (obrigatório para SVM).
4. Separamos a variável alvo `Risk_Flag`.


In [ ]:
# ── Separação de features e alvo ──────────────────────────────────────────────
DROP_COLS = ["Id"]
TARGET    = "Risk_Flag"

X_train = df_train.drop(columns=DROP_COLS + [TARGET])
y_train = df_train[TARGET]

X_test  = df_test.drop(columns=DROP_COLS + [TARGET])
y_test  = df_test[TARGET]

num_features = ["Income", "Age", "Experience", "CURRENT_JOB_YRS", "CURRENT_HOUSE_YRS"]
cat_features = ["Married/Single", "House_Ownership", "Car_Ownership",
                "Profession", "CITY", "STATE"]

print(f"X_train: {X_train.shape} | y_train: {y_train.shape}")
print(f"X_test : {X_test.shape}  | y_test : {y_test.shape}")


In [ ]:
# ── Construção do Pipeline de pré-processamento ───────────────────────────────
# O ColumnTransformer aplica diferentes transformações a cada grupo de colunas
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(),                                   num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_features),
    ]
)

# Ajuste do pré-processador SOMENTE no treino → aplicação no treino e no teste
preprocessor.fit(X_train)

X_train_proc = preprocessor.transform(X_train)
X_test_proc  = preprocessor.transform(X_test)

print(f"Shape após pré-processamento — Treino: {X_train_proc.shape} | Teste: {X_test_proc.shape}")


## 5. Busca em Grade de Hiperparâmetros (GridSearchCV)

Utilizamos **GridSearchCV** com **StratifiedKFold (5-fold)** para encontrar a melhor combinação de:

| Hiperparâmetro | Descrição | Valores testados |
|---|---|---|
| `kernel` | Função de mapeamento do espaço de features | `linear`, `rbf`, `poly` |
| `C` | Parâmetro de regularização | `0.1`, `1`, `10` |
| `gamma` | Coeficiente do kernel (rbf, poly) | `scale`, `auto` |

A métrica de otimização é o **F1-score (macro)**, pois o dataset é balanceado e queremos equilibrar precisão e revocação.

> ⚠️ O Grid Search é executado no conjunto de treino, **sem qualquer uso do conjunto de teste**.


In [ ]:
# ── Definição da grade de hiperparâmetros ─────────────────────────────────────
param_grid = {
    "kernel": ["linear", "rbf", "poly"],
    "C"     : [0.1, 1, 10],
    "gamma" : ["scale", "auto"],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

svm_search = GridSearchCV(
    estimator  = SVC(random_state=42),
    param_grid = param_grid,
    scoring    = "f1_macro",
    cv         = cv,
    n_jobs     = -1,
    verbose    = 1,
    refit      = True,   # refaz o treino com os melhores parâmetros em todo o conjunto
)

print("Iniciando GridSearchCV...")
t0_grid = time.time()
svm_search.fit(X_train_proc, y_train)
t_grid  = time.time() - t0_grid

print(f"\n✅ GridSearchCV concluído em {t_grid:.1f} s")
print(f"   Melhores parâmetros : {svm_search.best_params_}")
print(f"   Melhor F1-macro (CV): {svm_search.best_score_:.4f}")


In [ ]:
# ── Resultados completos da busca em grade ────────────────────────────────────
results_df = pd.DataFrame(svm_search.cv_results_)
cols_show  = ["param_kernel", "param_C", "param_gamma",
              "mean_test_score", "std_test_score", "rank_test_score"]
display(
    results_df[cols_show]
    .sort_values("rank_test_score")
    .rename(columns={
        "param_kernel"   : "kernel",
        "param_C"        : "C",
        "param_gamma"    : "gamma",
        "mean_test_score": "F1_macro_médio",
        "std_test_score" : "Desvio_padrão",
        "rank_test_score": "Rank",
    })
    .reset_index(drop=True)
    .head(10)
)


## 6. Avaliação do Melhor Modelo

O `GridSearchCV` com `refit=True` já refaz o treinamento automático com os melhores hiperparâmetros.  
Aqui medimos o tempo de **indução (treino final)** e de **predição no conjunto de teste**.


In [ ]:
# ── Tempo de indução do melhor modelo (refit já executado no GridSearch) ───────
best_svm = svm_search.best_estimator_
print(f"Melhor modelo: {best_svm}")
print()

# Medindo tempo de predição no TREINO
t0 = time.time()
y_train_pred = best_svm.predict(X_train_proc)
t_pred_train = time.time() - t0

# Medindo tempo de predição no TESTE
t0 = time.time()
y_test_pred  = best_svm.predict(X_test_proc)
t_pred_test  = time.time() - t0

print(f"⏱  Tempo de predição (treino): {t_pred_train:.4f} s")
print(f"⏱  Tempo de predição (teste) : {t_pred_test:.4f} s")
print(f"⏱  Tempo total de busca+indução (GridSearch): {t_grid:.1f} s")


## 7. Métricas de Avaliação

Calculamos **acurácia, F1-macro, precisão e revocação** tanto no treino quanto no teste  
para verificar overfitting e a qualidade preditiva do modelo.


In [ ]:
# ── Tabela de métricas ────────────────────────────────────────────────────────
def metricas(y_true, y_pred, nome):
    return {
        "Conjunto"  : nome,
        "Acurácia"  : accuracy_score(y_true, y_pred),
        "F1-macro"  : f1_score(y_true, y_pred, average="macro"),
        "Precisão"  : precision_score(y_true, y_pred, average="macro"),
        "Revocação" : recall_score(y_true, y_pred, average="macro"),
    }

metricas_df = pd.DataFrame([
    metricas(y_train, y_train_pred, "Treino"),
    metricas(y_test,  y_test_pred,  "Teste"),
])
metricas_df = metricas_df.set_index("Conjunto")
metricas_df = metricas_df.applymap(lambda x: f"{x:.4f}")
print("=== Métricas de Avaliação ===")
display(metricas_df)


In [ ]:
# ── Relatório de classificação completo (Teste) ───────────────────────────────
print("=== Classification Report (Teste) ===")
print(classification_report(y_test, y_test_pred,
                            target_names=["Baixo risco (0)", "Alto risco (1)"]))


In [ ]:
# ── Matriz de confusão (Teste) ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_test_pred,
    display_labels=["Baixo risco (0)", "Alto risco (1)"],
    cmap="Blues", ax=ax
)
ax.set_title("Matriz de Confusão – Conjunto de Teste")
plt.tight_layout()
plt.show()


## 8. Fronteira de Decisão (DecisionBoundaryDisplay)

Para visualizar a fronteira de decisão, selecionamos os **2 atributos mais correlacionados** com  
`Risk_Flag` (identificados na análise exploratória): **Experience** e **House_Ownership**.

Treinamos um SVM auxiliar **apenas com esses 2 atributos** (após pré-processamento específico  
para esses atributos) para permitir a visualização 2D da fronteira.


In [ ]:
# ── Pré-processamento só para os 2 top atributos ─────────────────────────────
# TOP2 definido na EDA: ['Experience', 'House_Ownership']
# House_Ownership é categórico → OHE; Experience é numérico → Scaler

top2_num = [c for c in TOP2 if c in num_features]
top2_cat = [c for c in TOP2 if c in cat_features]

print(f"Top-2 atributos: {TOP2}")
print(f"  Numéricos : {top2_num}")
print(f"  Categóricos: {top2_cat}")

prep_top2 = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), top2_num),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), top2_cat),
    ] if top2_cat else [
        ("num", StandardScaler(), top2_num),
    ]
)

X_train_top2 = prep_top2.fit_transform(X_train)
X_test_top2  = prep_top2.transform(X_test)
print(f"Shape Top-2 – Treino: {X_train_top2.shape} | Teste: {X_test_top2.shape}")


In [ ]:
# ── Treino do SVM auxiliar (top-2 atributos, melhores hiperparâmetros) ────────
best_params = svm_search.best_params_
svm_2d = SVC(**best_params, random_state=42)

t0 = time.time()
svm_2d.fit(X_train_top2, y_train)
t_2d = time.time() - t0
print(f"SVM 2D treinado em {t_2d:.4f} s | Parâmetros: {best_params}")

# Nota: para Decision Boundary 2D precisamos de exatamente 2 features
# Se House_Ownership gerou mais colunas via OHE, reduzimos às 2 primeiras (componentes)
if X_train_top2.shape[1] > 2:
    from sklearn.decomposition import PCA
    pca = PCA(n_components=2, random_state=42)
    X_train_top2 = pca.fit_transform(X_train_top2)
    X_test_top2  = pca.transform(X_test_top2)
    svm_2d = SVC(**best_params, random_state=42)
    svm_2d.fit(X_train_top2, y_train)
    xlabel, ylabel = "Componente PCA 1 (Experience + House_Ownership)", "Componente PCA 2"
    print("PCA aplicado para redução a 2D.")
else:
    xlabel, ylabel = TOP2[0], TOP2[1]


In [ ]:
# ── DecisionBoundaryDisplay – Treino e Teste ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, X_2d, y_true, titulo in [
    (axes[0], X_train_top2, y_train, "Conjunto de Treino"),
    (axes[1], X_test_top2,  y_test,  "Conjunto de Teste"),
]:
    disp = DecisionBoundaryDisplay.from_estimator(
        svm_2d, X_2d,
        response_method="predict",
        alpha=0.3,
        ax=ax,
        cmap="RdYlGn",
    )
    scatter = ax.scatter(
        X_2d[:, 0], X_2d[:, 1],
        c=y_true, cmap="RdYlGn",
        edgecolors="k", s=10, alpha=0.6,
        vmin=0, vmax=1
    )
    ax.set_title(f"Fronteira de Decisão SVM – {titulo}")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)

plt.colorbar(scatter, ax=axes, label="Risk_Flag (0=baixo, 1=alto)")
plt.suptitle(f"Top-2 atributos: {TOP2[0]} e {TOP2[1]}", fontsize=12)
plt.tight_layout()
plt.show()


## 9. Resumo dos Tempos de Execução


In [ ]:
# ── Tabela de tempos ──────────────────────────────────────────────────────────
tempos_df = pd.DataFrame({
    "Etapa": [
        "GridSearchCV (busca + re-indução final)",
        "Predição – Treino",
        "Predição – Teste",
        "Indução SVM 2D (top-2 atributos)",
    ],
    "Tempo (s)": [
        f"{t_grid:.2f}",
        f"{t_pred_train:.4f}",
        f"{t_pred_test:.4f}",
        f"{t_2d:.4f}",
    ]
})
display(tempos_df)


## 10. Análise e Interpretação dos Resultados

### 10.1 Qualidade da Predição vs. Custo de Treinamento

- O SVM com kernel **RBF/linear/poly** (dependendo do melhor encontrado pelo Grid Search) é capaz de  
  generalizar bem neste dataset balanceado.
- **Acurácia e F1** próximos entre treino e teste indicam boa capacidade de generalização (sem overfitting severo).
- O parâmetro **C** controla o trade-off entre margem e erros de classificação:  
  - C pequeno → margem larga, mais erros tolerados (maior bias, menor variância);  
  - C grande → margem estreita, menos erros tolerados (menor bias, maior variância).

### 10.2 Custo Computacional

- O **Grid Search** é o passo mais custoso: avalia múltiplas combinações de hiperparâmetros via  
  validação cruzada estratificada (5-fold). O tempo cresce proporcionalmente ao número de combinações testadas.
- A **predição** é muito rápida em relação à indução, tornando o modelo adequado para inferência em produção.
- Kernels como **RBF** e **poly** têm custo quadrático ou cúbico no número de vetores de suporte,  
  o que pode ser relevante em datasets muito maiores.

### 10.3 Atributos Mais Correlacionados com Risk_Flag

Os dois atributos com maior correlação (absoluta) com a variável alvo foram:  
- **Experience** (anos de experiência profissional): correlação negativa — mais experiência, menor risco.  
- **House_Ownership** (situação habitacional): indica perfil financeiro do solicitante.

Esses atributos foram usados para plotar as fronteiras de decisão 2D.

### 10.4 LLM Utilizada

Este notebook foi desenvolvido com auxílio da LLM **Claude Sonnet 4.6** (Anthropic), via claude.ai.  
Sequência de prompts:
1. *"Verifiquem o que é pedido no enunciado, e entreguem um notebook python formato ipynb para rodar no colab."*
2. Iterações de ajuste de análise exploratória e verificação dos dados.
